# 07_eval_interpret — Evaluation & Interpretation

The project's original question: **"What drives the multiple up?"**

Using the best model (Tabular + Text GradientBoosting from 06), we:
1. Rank feature importances — what matters most
2. Measure how much the TEXT block contributes vs tabular features
3. SHAP summary — direction of each feature's effect
4. Residual analysis — where does the model err?
5. Error by segment (profit scale / monetization)

> Runs locally. Needs both `features_*.csv` and `text_embeddings_*.csv` in data/processed/.


In [ ]:
import ast
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import GradientBoostingRegressor
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_predict, KFold

warnings.filterwarnings("ignore")

PROJECT_ROOT = Path.cwd().parents[1] if (Path.cwd().name == "local") else Path.cwd()
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

feat = pd.read_csv(sorted(PROCESSED_DIR.glob("features_*.csv"))[-1])
emb = pd.read_csv(sorted(PROCESSED_DIR.glob("text_embeddings_*.csv"))[-1])
df = feat.merge(emb, on="id", how="inner").copy()
print("Merged:", df.shape)

TARGET_LOG = "log_target"
TARGET_RAW = "annual_listing_multiple"


## 1. Build the Best Model (Tabular + Text GBR)

Same feature set and seed as 06. We fit on all data for interpretation,
and separately get cross-validated predictions for honest residual analysis.


In [ ]:
def first_of(x, key):
    try:
        lst = ast.literal_eval(x) if isinstance(x, str) else x
        if isinstance(lst, list) and lst:
            item = lst[0]
            return item.get(key) if isinstance(item, dict) else item
    except Exception:
        pass
    return "Unknown"

df["primary_monetization"] = df["monetizations"].apply(lambda x: first_of(x, "monetization"))
df["primary_niche"] = df["niches"].apply(lambda x: first_of(x, "niche"))

num = ["log_average_annual_net_profit", "log_average_annual_gross_revenue",
       "business_age_months", "net_margin", "hours_worked_per_week",
       "days_on_marketplace", "monetizations_count", "niches_count", "amazon_sku_count"]
boo = ["has_trademark", "uses_pbn", "private_lender_approved",
       "patent_pending", "patented_design", "patented_utility"]
cat = ["primary_monetization", "primary_niche", "country", "source"]
text = [c for c in df.columns if c.startswith("text_emb_")]

for c in num + boo + text:
    df[c] = pd.to_numeric(df[c], errors="coerce")

X = df[num + boo + text + cat].copy()
y = df[TARGET_LOG].values
y_raw = df[TARGET_RAW].values

pre = ColumnTransformer([
    ("num", SimpleImputer(strategy="median"), num + boo + text),
    ("cat", OneHotEncoder(handle_unknown="ignore", min_frequency=5), cat),
])
pipe = Pipeline([("pre", pre), ("model", GradientBoostingRegressor(random_state=42))])

# Cross-validated predictions (for residuals)
cv = KFold(n_splits=5, shuffle=True, random_state=42)
pred_log_cv = cross_val_predict(pipe, X, y, cv=cv)

# Fit on all data (for feature importance / SHAP)
pipe.fit(X, y)

# Recover feature names after one-hot
ohe = pipe.named_steps["pre"].named_transformers_["cat"]
cat_names = list(ohe.get_feature_names_out(cat))
feature_names = num + boo + text + cat_names
print("Total features after encoding:", len(feature_names))


## 2. Feature Importance — What Matters Most

Gradient boosting importances. We also collapse the 50 text embedding dims
into a single "TEXT (all dims)" bar so text and tabular are comparable.


In [ ]:
importances = pd.Series(pipe.named_steps["model"].feature_importances_, index=feature_names)

# Collapse text dims into one bucket for a fair tabular-vs-text view
text_total = importances[[c for c in importances.index if c.startswith("text_emb_")]].sum()
tabular_imp = importances[[c for c in importances.index if not c.startswith("text_emb_")]]
collapsed = tabular_imp.copy()
collapsed["TEXT (all 50 dims)"] = text_total
collapsed = collapsed.sort_values(ascending=False).head(15)

fig, ax = plt.subplots(figsize=(9, 6))
collapsed[::-1].plot(kind="barh", ax=ax, color="#4C72B0")
ax.set_title("Top feature importances (text dims collapsed)")
ax.set_xlabel("importance")
plt.tight_layout()
plt.show()

print(f"TEXT block total importance: {text_total:.3f}")
print(f"Tabular total importance:    {1 - text_total:.3f}")


## 3. SHAP Summary — Direction of Effects

Feature importance says *how much* a feature matters; SHAP says *which way*.
E.g. does higher net profit push the predicted multiple up or down?
(We sample rows for speed.)


In [ ]:
import shap

X_trans = pipe.named_steps["pre"].transform(X)
if hasattr(X_trans, "toarray"):
    X_trans = X_trans.toarray()

sample_n = min(500, X_trans.shape[0])
idx = np.random.RandomState(42).choice(X_trans.shape[0], sample_n, replace=False)

explainer = shap.TreeExplainer(pipe.named_steps["model"])
shap_values = explainer.shap_values(X_trans[idx])

# Focus the plot on non-text features for readability
non_text_mask = [not n.startswith("text_emb_") for n in feature_names]
non_text_idx = [i for i, keep in enumerate(non_text_mask) if keep]

shap.summary_plot(
    shap_values[:, non_text_idx],
    X_trans[idx][:, non_text_idx],
    feature_names=[feature_names[i] for i in non_text_idx],
    show=True,
    max_display=15,
)


## 4. Residual Analysis — Where Does It Err?

Using cross-validated predictions (honest, not fit on the same rows).
Residual = actual - predicted (in original multiple units).
- Points far from 0 are big misses.
- A funnel shape means error grows with the predicted value.


In [ ]:
pred_raw_cv = np.expm1(pred_log_cv)
residual = y_raw - pred_raw_cv

fig, ax = plt.subplots(1, 2, figsize=(12, 4))

ax[0].scatter(pred_raw_cv, residual, s=8, alpha=0.3, color="#C44E52")
ax[0].axhline(0, color="black", lw=1)
ax[0].set_title("Residual vs predicted")
ax[0].set_xlabel("predicted multiple")
ax[0].set_ylabel("residual (actual - predicted)")

ax[1].hist(residual, bins=50, color="#55A868", edgecolor="white")
ax[1].axvline(0, color="black", lw=1)
ax[1].set_title("Residual distribution")
ax[1].set_xlabel("residual")

plt.tight_layout()
plt.show()

print("Residual stats:", pd.Series(residual).describe().round(3).to_dict())


## 5. Error by Segment

Does the model do better on some kinds of businesses than others?
Break down mean absolute error by profit scale and by monetization type.


In [ ]:
diag = pd.DataFrame({
    "actual": y_raw,
    "pred": pred_raw_cv,
    "abs_err": np.abs(residual),
    "primary_monetization": df["primary_monetization"].values,
    "net_profit": pd.to_numeric(df["average_annual_net_profit"], errors="coerce").values,
})

# By profit scale (quartiles)
diag["profit_bucket"] = pd.qcut(diag["net_profit"], 4,
                                labels=["Q1 (small)", "Q2", "Q3", "Q4 (large)"])
by_profit = diag.groupby("profit_bucket")["abs_err"].mean()

# By monetization (types with >= 30 listings)
counts = diag["primary_monetization"].value_counts()
common = counts[counts >= 30].index
by_monet = diag[diag["primary_monetization"].isin(common)].groupby("primary_monetization")["abs_err"].mean().sort_values()

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
by_profit.plot(kind="bar", ax=ax[0], color="#4C72B0")
ax[0].set_title("Mean abs error by profit scale")
ax[0].set_ylabel("MAE (multiple)")
ax[0].tick_params(axis="x", rotation=0)

by_monet.plot(kind="barh", ax=ax[1], color="#55A868")
ax[1].set_title("Mean abs error by monetization (n>=30)")
ax[1].set_xlabel("MAE (multiple)")

plt.tight_layout()
plt.show()


## 6. Takeaways (fill in after running)

- Which features drive the multiple most? (net profit / revenue scale, age, monetization, text?)
- How much does the TEXT block contribute overall?
- SHAP directions: which factors push multiples up vs down?
- Where is the model least reliable (which profit scale / monetization)?

**Next: `08_report`** — write up derived/aggregate results + code + short quotes only.
Raw text stays private.
